# 00 · 准备响应表示与图谱输入

**问题：Figure 1 读取的 means、SNR、template、coefficient 和排序是怎样生成的？** 这本 notebook 把此前藏在历史探索脚本里的准备步骤放到可阅读、可重跑的位置，真正计算输入文件，再交给 [01_response_atlas](01_response_atlas.ipynb) 绘图。

起点是已审核的 **个体平均 ΔF/F₀ 曲线缓存**，其中同一动物的 trials 已经等权平均；不是显微镜荧光原始表。这里不重做分割、ΔF/F₀ 预处理或 trial QC。我们核对个体曲线与已审核的时间箱及基线统计一致，然后按已接受的固定参数重建表示。

输出：`poster/data/prepared/responses/`。历史报告只读，旧结果用于一致性检查，不用于填充新结果。Split-half / LOAO、化学输入及 HMDS 是另外的分析阶段，不在本册重新计算。

In [ ]:
from pathlib import Path
import sys
import hashlib
import json
from datetime import datetime, timezone
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'poster/src/poster_analysis').is_dir() and (p / 'reports').is_dir())
sys.path.insert(0, str(ROOT / 'poster/src'))
from poster_analysis.constants import NEURONS
from poster_analysis.paths import report_path, table_dir
from poster_analysis.style import apply_style
from poster_analysis.preparation import (
    load_animal_inputs, bin_curves, fit_representation,
    aggregate_strains, tables_for_fits, build_display_tables,
)
apply_style()

# Accepted poster settings. Historical parity below deliberately checks this definition.
PRIMARY_THRESHOLD = 0.5
MIN_ANIMALS = 2
SENSITIVITY_THRESHOLDS = (None, 0.25, 0.5, 0.75, 1.0)
PREPARED = ROOT / 'poster/data/prepared/responses'
REFERENCE = report_path('responses', ROOT)
EXPORT_TABLES = table_dir(ROOT)

## 1. 上游输入与数据维度

前两份文件提供个体平均曲线及辅助基线噪声。Trial 缓存只读取 −5 至 −1 s 的五个刺激前样本，用于独立核对“每 trial 的 SD → 每动物的中位数”；审核过的 observations 用来核对 5-s 分箱数值与缺失位置。

整理后的数组为 **animal × condition × neuron × 40 个一秒样本**。Condition 是菌株和记录批次的唯一组合，仅用于保存重复测量结构和既定等权汇总。未记录位置是 NaN，不能填零。

In [ ]:
source_paths = {
    'animal_curves': ROOT / 'reports/population/exploration_20260929/tables/animal_curves.parquet',
    'animal_metrics': ROOT / 'reports/population/exploration_20260929/tables/animal_metrics.csv',
    'trial_curves': ROOT / 'reports/population/exploration_20260929/tables/trial_curves.parquet',
    'observations': report_path('response_structure', ROOT) / 'data/observations.parquet',
}
source_index = pd.DataFrame([
    {'role': name, 'path': str(path.relative_to(ROOT)),
     'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    for name, path in source_paths.items()
])
display(source_index[['role', 'path']])
inputs = load_animal_inputs(
    source_paths['animal_curves'], source_paths['animal_metrics'],
    source_paths['trial_curves'], source_paths['observations'], cells=NEURONS,
)
raw, baseline = inputs['raw'], inputs['baseline_sd']
conditions, neurons = np.asarray(inputs['conditions']), inputs['cells']
condition_strains = conditions[:, 0]
display(pd.Series({k: v for k, v in inputs['metadata'].items() if k != 'input_sha256'}, name='value'))
assert raw.shape == (49, 112, 13, 40)
assert np.isfinite(raw).all(axis=-1).sum() == 7063
for row in source_index.itertuples(index=False):
    assert inputs['metadata']['input_sha256'][row.role] == row.sha256

## 2. 在一组真实曲线上手算 individual SNR

每个 condition–neuron 用 n 只动物的曲线求平均 μ(t)，然后计算：

- P = meanₜ[μ(t)²]：平均曲线的功率；
- V = meanₜ[varₐₙᵢₘₐₗ(x(t), ddof=1)]：动物之间的散布；
- C = P − V/n，SNR = √[max(C, 0) / V]。

这些量使用完整 0–40 s，不先做 neuron z-score。最少两只动物，SNR ≥ 0.5 时保留。V=0 时，C>0 对应无穷大，C=0 对应零。下面用同一 A300–AWCON 例子展示计算，不用这个例子选择阈值。基线噪声仅作辅助描述，不参与门控。

In [ ]:
c = neurons.index('AWCON')
k = np.flatnonzero(condition_strains == 'A300')[0]
curves = raw[:, k, c]
curves = curves[np.isfinite(curves).all(axis=1)]
n = len(curves)
mu = curves.mean(axis=0)
P = np.mean(mu ** 2)
V = np.var(curves, axis=0, ddof=1).mean()
C = P - V / n
snr_example = np.sqrt(max(C, 0) / V) if V > 0 else (np.inf if C > 0 else 0.0)
display(pd.Series({'n_animals': n, 'P': P, 'V': V, 'C': C,
                   'SNR': snr_example, 'passes_gate': n >= MIN_ANIMALS and snr_example >= PRIMARY_THRESHOLD}))
fig, ax = plt.subplots(figsize=(7, 3.1), layout='constrained')
ax.plot(np.arange(40), curves.T, color='#99A8B8', alpha=.55, lw=.8)
ax.plot(np.arange(40), mu, color='#26354A', lw=2, label='Equal-animal mean')
ax.axvspan(0, 10, color='#D8DEE5', alpha=.3)
ax.set(xlabel='Time (s)', ylabel=r'$\Delta F/F_0$', title=f'A300 · AWCON · {n} animal means')
ax.legend(frameon=False)
plt.show()

## 3. 5-s 分箱、共享模板和投影系数

每条 40-s 曲线取八个连续 5-s 均值。每类神经元用通过门控的条件均值拟合一个**不中心化的加权 rank-1 SVD**：每个菌株等权，该菌株的可用条件再等分它的权重。

模板 h 的 RMS 设为 1，绝对值最大的 bin 固定为正；系数 a = y·h / (h·h)，重建为 a×h。系数的正负是相对模板方向的符号，不普遍等价于兴奋/抑制。下面先计算全部表示，再独立展开 AWCON 的加权 SVD 核对算法。

In [ ]:
main = fit_representation(raw, baseline, condition_strains,
                          threshold=PRIMARY_THRESHOLD, min_animals=MIN_ANIMALS)
np.testing.assert_allclose(main['snr'][k, c], snr_example, atol=1e-12)
np.testing.assert_allclose(mu.reshape(8, 5).mean(axis=1), main['mean_bins'][k, c], atol=1e-12)

kept = np.flatnonzero((main['counts'][:, c] >= MIN_ANIMALS) &
                      (main['snr'][:, c] >= PRIMARY_THRESHOLD))
y = main['mean_bins'][kept, c]
unique_strains, inverse, n_blocks = np.unique(condition_strains[kept], return_inverse=True, return_counts=True)
weights = 1 / (len(unique_strains) * n_blocks[inverse])
_, _, vt = np.linalg.svd(np.sqrt(weights[:, None]) * y, full_matrices=False)
h = vt[0] / np.sqrt(np.mean(vt[0] ** 2))
if h[np.argmax(np.abs(h))] < 0:
    h = -h
amplitudes = y @ h / (h @ h)
np.testing.assert_allclose(h, main['templates'][c], atol=1e-12)
np.testing.assert_allclose(amplitudes, main['coefficients'][kept, c], atol=1e-12)
np.testing.assert_allclose(np.sqrt((main['templates'] ** 2).mean(axis=1)), 1, atol=1e-12)
display(pd.DataFrame({'bin_start_s': np.arange(0, 40, 5), 'AWCON_template': h}))
display(pd.Series(dict(zip(*np.unique(main['status'], return_counts=True))), name='condition_neuron_count'))

零与缺失必须分开：满足动物数但未过 SNR 门槛的条件，系数和重建置零；动物数不足或缺失保留 NaN。若全零均值不能识别模板，保留 `observed_zero` / `template_unidentified` 等状态。Notebook 01 只负责展示这些已经准备好的值。

## 4. 条件到菌株，再构造显示排序

同菌株的有限条件系数等权平均，得到 106×13 的 strain coefficients。排序独立使用**未门控的八箱原始均值**：先要求至少两只动物，然后按菌株等权汇总；每对菌株在共同完整神经元上计算 cosine，至少共同 4 类神经元。距离是 1−cosine，完整矩阵用 average linkage 排序，缺失距离则保留菌株 ID 顺序，不插补。

显示值从已门控的完整八箱重建截取前五箱；排序不根据化学数据或这张图的颜色来挑选。

In [ ]:
strain_values, strain_ids = aggregate_strains(main['coefficients'], conditions)
strain_coefficients = pd.DataFrame(strain_values, index=pd.Index(strain_ids, name='strain'), columns=neurons)
display(strain_coefficients.iloc[:5, :6])
display_tables = build_display_tables(main, conditions, neurons)
raw_profiles, _ = aggregate_strains(
    np.where(main['eligible'][..., None], main['mean_bins'], np.nan), conditions)
# One explicit pair, including the shared-cell rule.
i, j = 0, 1
shared = np.isfinite(raw_profiles[[i, j]]).all(axis=(0, 2))
left, right = raw_profiles[i, shared].ravel(), raw_profiles[j, shared].ravel()
assert shared.sum() >= 4
pair_distance = 1 - np.dot(left, right) / (np.linalg.norm(left) * np.linalg.norm(right))
np.testing.assert_allclose(pair_distance, display_tables['rdm_raw'].iloc[i, j], atol=1e-12)
# Display the hierarchy step directly as well.
d = display_tables['rdm_raw'].to_numpy()
assert np.isfinite(d).all(), 'This canonical poster dataset has complete raw-profile distances'
d = np.clip((d + d.T) / 2, 0, 2)
np.fill_diagonal(d, 0)
leaf_order = leaves_list(linkage(squareform(d), method='average'))
assert [strain_ids[i] for i in leaf_order] == display_tables['figure_row_order'].strain.tolist()
display(display_tables['figure_row_order'].head(8))
print(display_tables['row_order_rule'])

## 5. 固定阈值的敏感性准备

分别拟合未门控、0.25、0.5、0.75、1.0 五种条件；每个阈值都重新选择用于拟合的条件并重新拟合模板，不能只对同一套系数做裁剪。这里只复现已使用的阈值，不搜索化学相关性最强的阈值。Split-half / LOAO 涉及训练集内重拟合，是另外的缓存验证阶段。

In [ ]:
fits = {}
for threshold in SENSITIVITY_THRESHOLDS:
    label = 'unfiltered' if threshold is None else f'{threshold:g}'
    fits[label] = main if threshold == PRIMARY_THRESHOLD else fit_representation(
        raw, baseline, condition_strains, threshold=threshold, min_animals=MIN_ANIMALS)
fit_tables = tables_for_fits(fits, conditions, neurons, primary_label=f'{PRIMARY_THRESHOLD:g}')
pd.testing.assert_frame_equal(fit_tables['strain_coefficients'], strain_coefficients)
display(fit_tables['sensitivity_metrics'].groupby('threshold')[['eligible', 'retained', 'suppressed', 'limited_n', 'missing']].sum())

## 6. 与历史结果核对，再发布准备文件

这里才读取历史结果作为外部数值基准。以上拟合、距离、排序和表格没有从这些结果复制。所有数组、门控状态、排序与旧表都应一致，浮点和 CSV 往返只允许舍入量级差异；参数仍是当前 poster 的固定定义。若有意探索不同参数，应使用独立输出位置并重新决定下游比较范围。

In [ ]:
checks = []
computed_arrays = {key: value for key, value in main.items() if isinstance(value, np.ndarray)}
computed_arrays.update(cells=np.asarray(neurons), conditions=conditions)
with np.load(REFERENCE / 'tables/representation_arrays.npz', allow_pickle=False) as previous:
    assert set(previous.files) == set(computed_arrays)
    for name, values in computed_arrays.items():
        if values.dtype.kind in 'SUb':
            np.testing.assert_array_equal(values, previous[name])
            difference = 0.0
        else:
            np.testing.assert_allclose(values, previous[name], rtol=1e-10, atol=1e-12, equal_nan=True)
            finite = np.isfinite(values) & np.isfinite(previous[name])
            difference = float(np.max(np.abs(values[finite] - previous[name][finite]))) if finite.any() else 0.0
        checks.append({'item': f'array:{name}', 'passed': True, 'max_abs_difference': difference})

output_tables = {**fit_tables, **{name: value for name, value in display_tables.items() if isinstance(value, pd.DataFrame)}}
indexed_tables = {'strain_coefficients', 'rdm_raw', 'rdm_raw_shared_cells', 'display_profile_5bin'}
for name, current in output_tables.items():
    kwargs = {'index_col': 0} if name in indexed_tables else {}
    dtype = {col: str for col in ('block', 'threshold') if col in current.columns}
    previous = pd.read_csv(REFERENCE / f'tables/{name}.csv', dtype=dtype or None, **kwargs)
    pd.testing.assert_frame_equal(current, previous, check_dtype=False, rtol=1e-10, atol=1e-12)
    checks.append({'item': f'table:{name}', 'passed': True, 'max_abs_difference': None})
check_table = pd.DataFrame(checks)
display(check_table)

以下输出目录中的文件是本次计算的结果。`representation_parameters.json` 记录方法与实际输入；`manifest.json` 记录每个输出的指纹并最后写入，供 Notebook 01 检查准备是否完整。原 `reports/` 文件不会被覆盖。

最初六项来源还应加上 `condition_metrics.csv` 和 `filter_state_by_strain_cell.csv`；它们也是 atlas loader 的输入。本册同时保存完整模板、敏感性明细、raw distance 和支持矩阵，便于追溯排序与状态。

In [ ]:
parameters = {
    'snr_basis': 'individual', 'primary_threshold': PRIMARY_THRESHOLD,
    'sensitivity_thresholds': list(SENSITIVITY_THRESHOLDS), 'min_animals': MIN_ANIMALS,
    'trial_count_gate': False, 'raw_window_seconds': [0, 40], 'stimulus_seconds': [0, 10],
    'model_bin_seconds': 5, 'display_bins': 5, 'cells': neurons, 'response_units': 'delta_F_over_F0',
    'gate': 'P=mean_t(mu^2); V=mean_t(var_animal(ddof=1)); C=P-V/n; SNR=sqrt(max(C,0)/V)',
    'zero_scatter': 'Positive coherent power gives inf SNR; zero coherent power gives zero SNR',
    'template_fit': 'Uncentred weighted SVD; equal strains then equal retained conditions within strain',
    'template_normalization': 'RMS=1; largest-absolute bin positive, first bin breaks exact tie',
    'strain_aggregation': 'Finite-value mean of available condition blocks within strain; no imputation',
    'coefficient_zero': 'Adequately observed gate failure or measured zero; missing/limited_n remains NaN',
    'baseline_snr': 'Auxiliary only, never used for gate',
    'row_order_rule': display_tables['row_order_rule'], 'min_shared_neurons_for_order': 4,
    'description_only': 'Full-data templates are descriptive; validation must refit training subsets',
    'prepared_from': 'Reviewed individual animal-mean curves, not raw fluorescence',
    'preparation_notebook': 'poster/notebooks/00_preparation.ipynb',
    'input_sha256': dict(zip(source_index.path, source_index.sha256)),
    'input_checks': {k: v for k, v in inputs['metadata'].items() if k != 'input_sha256'},
    'code_sha256': hashlib.sha256((ROOT / 'poster/src/poster_analysis/preparation.py').read_bytes()).hexdigest(),
}
# Compare scientific settings too; provenance fields intentionally describe this new preparation.
reference_parameters = json.loads((REFERENCE / 'representation_parameters.json').read_text())
for key in ['snr_basis', 'primary_threshold', 'sensitivity_thresholds', 'min_animals',
            'trial_count_gate', 'raw_window_seconds', 'stimulus_seconds', 'model_bin_seconds', 'cells']:
    assert parameters[key] == reference_parameters[key], key
reference_paths = [REFERENCE / 'tables/representation_arrays.npz', REFERENCE / 'representation_parameters.json']
reference_paths += [REFERENCE / f'tables/{name}.csv' for name in output_tables]
PREPARED.mkdir(parents=True, exist_ok=True)
(PREPARED / 'tables').mkdir(exist_ok=True)
# Invalidate any previous publication marker before writing the new bundle.
(PREPARED / 'manifest.json').unlink(missing_ok=True)
np.savez_compressed(PREPARED / 'tables/representation_arrays.npz', **computed_arrays)
for name, frame in output_tables.items():
    frame.to_csv(PREPARED / f'tables/{name}.csv', index=name in indexed_tables)
(PREPARED / 'representation_parameters.json').write_text(json.dumps(parameters, ensure_ascii=False, indent=2) + '\n')
output_paths = [PREPARED / 'tables/representation_arrays.npz', PREPARED / 'representation_parameters.json']
output_paths += [PREPARED / f'tables/{name}.csv' for name in output_tables]
manifest = {
    'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
    'producer': 'poster/notebooks/00_preparation.ipynb',
    'source_sha256': parameters['input_sha256'],
    'reference_sha256': {str(path.relative_to(ROOT)): hashlib.sha256(path.read_bytes()).hexdigest() for path in reference_paths},
    'output_sha256': {str(path.relative_to(PREPARED)): hashlib.sha256(path.read_bytes()).hexdigest() for path in output_paths},
    'historical_parity_checks': len(check_table), 'all_checks_passed': bool(check_table.passed.all()),
}
# Ensure upstream files did not change during preparation.
for row in source_index.itertuples(index=False):
    assert hashlib.sha256((ROOT / row.path).read_bytes()).hexdigest() == row.sha256
source_index.to_csv(EXPORT_TABLES / 'prep00_sources.csv', index=False)
check_table.to_csv(EXPORT_TABLES / 'prep00_checks.csv', index=False)
(PREPARED / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n')
from poster_analysis.paths import prepared_response_path
assert prepared_response_path(ROOT) == PREPARED
display(pd.DataFrame({'prepared_file': list(manifest['output_sha256'])}))

## 后续怎样使用

`00_preparation → poster/data/prepared/responses/ → 01_response_atlas`。

图 1 的版式修改只需重跑 01；更新上游曲线或准备实现时，先重跑 00 并检查一致性，再运行 01。这里的元数据和历史断言固定了目前采用的方法，不会悄悄改动筛选定义。

Notebook 02 的重复性与留出表仍来自独立验证缓存；Notebook 03–04 的化学输入仍来自当前 162 项报告。这本 preparation 解决的是图 1 这批输入的生成，不表示全部分析都已经从原始数据迁入 poster。